# P3 — Paths are messages

This connects P1 (over-squashing) and P2 (counting paths). After `g` layers, a message from `i` reaches `j`
**once per length-g path**. So `n_g(i,j) = (A^g)[i,j]` is exactly how many copies of `i`'s information land
at `j` — all forced into one fixed vector.

## 1. Each path delivers one message

<img src="../figs-theory/en/E3a_path_is_message.svg" width="760"/>

## 2. Messages stack into one vector

When the stack exceeds the vector's capacity, the receiver cannot disentangle them — over-squashing.

<img src="../figs-theory/en/E3b_messages_stack.svg" width="760"/>

In [ ]:
# Count raw messages vs the de-duplicated (quotient) count into the target.
import torch
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators
K, M = 5, 4
for d in [1,2,3]:
    dd = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, eff = walk_operators(dd.edge_index.numpy(), dd.num_nodes, max_length=d+1)
    t = int(dd.root_mask.nonzero()[0])
    print(f'depth {d}: raw {int(raw[d][:,t].sum()):4d}   de-duplicated (kQ/I) {int(eff[d][:,t].sum()):3d}')

## 3. The quotient kQ/I: collapsing equivalent paths

Imposing a relation merges parallel paths into one class, lowering the multiplicity.

<img src="../figs-theory/en/T5_quotient.svg" width="760"/>

In [ ]:
# kQ/I collapses the diamond's two length-2 paths from 2 to 1 (via aiq-quivers).
from oversquash.ideal_bridge import build_quotient_plan
import numpy as np
ei = np.array([[0,0,1,2],[1,2,3,3]])   # 0->1, 0->2, 1->3, 2->3
plan = build_quotient_plan(ei, num_nodes=4, max_length=2)
print('raw multiplicity (0->3, length 2):', plan.raw_mult.get((0,3,2)))
print('effective (kQ/I)                 :', plan.effective_mult.get((0,3,2)))

## 4. The twist: when merging helps, and when it hurts

Collapsing parallel paths only helps when they are truly redundant. In our retrieval task the multiplicity
**is signal**, so collapsing it removes information — a negative result we confirm in the experiments.

<img src="../figs-theory/en/E3c_signal_vs_noise.svg" width="760"/>

**Next (P4):** keep every path, but *learn* how much to weight each one. That is attention.